# Notebook 5: DeepSeek Sparse Attention (DSA) - JAX

In [ ]:
import jax.numpy as jnp
from flax import linen as nn
from jax import random

class DeepSeekSparseAttention(nn.Module):
    """DSA in Flax"""
    hidden_size: int
    num_heads: int
    num_selected: int = 16
    
    def setup(self):
        self.head_dim = self.hidden_size // self.num_heads
        self.W_q = nn.Dense(self.hidden_size)
        self.W_KV = nn.Dense(self.head_dim * self.num_heads)
        self.W_o = nn.Dense(self.hidden_size)
    
    def __call__(self, x, use_sparse=True):
        batch, seq_len, _ = x.shape
        
        Q = self.W_q(x).reshape(batch, seq_len, self.num_heads, self.head_dim)
        Q = jnp.transpose(Q, (0, 2, 1, 3))
        
        KV = self.W_KV(x)
        K = KV.reshape(batch, seq_len, self.num_heads, self.head_dim)
        K = jnp.transpose(K, (0, 2, 1, 3))
        V = K  # Simplified
        
        if use_sparse:
            # Select top-k tokens
            scores = jnp.einsum('bhqd,bhkd->bhqk', Q, K)
            topk_scores, topk_idx = jax.lax.top_k(scores.reshape(batch * self.num_heads, seq_len, seq_len), self.num_selected)
            topk_idx = topk_idx.reshape(batch, self.num_heads, seq_len, self.num_selected)
            topk_scores = topk_scores.reshape(batch, self.num_heads, seq_len, self.num_selected)
            attn_weights = nn.softmax(topk_scores, axis=-1)
            
            # Simplified output
            attn_output = jnp.einsum('bhqk,bhvd->bhqd', attn_weights, V)
        else:
            scale = jnp.sqrt(self.head_dim)
            scores = jnp.einsum('bhqd,bhkd->bhqk', Q, K) / scale
            mask = jnp.tril(jnp.ones((seq_len, seq_len)))
            scores = jnp.where(mask == 0, -1e9, scores)
            attn_weights = nn.softmax(scores, axis=-1)
            attn_output = jnp.einsum('bhqk,bhvd->bhqd', attn_weights, V)
        
        attn_output = jnp.transpose(attn_output, (0, 2, 1, 3))
        attn_output = attn_output.reshape(batch, seq_len, self.hidden_size)
        return self.W_o(attn_output)

dsa = DeepSeekSparseAttention(hidden_size=512, num_heads=8, num_selected=16)
rng = random.PRNGKey(42)
x = random.normal(rng, (2, 32, 512))
params = dsa.init(rng, x)
output = dsa.apply(params, x, use_sparse=True)
print(f"DSA input: {x.shape}")
print(f"DSA output: {output.shape}")

## Verification
1. ✅ Can implement DSA in Flax
2. ✅ Understand sparse attention concept
3. ✅ Can use jax.lax.top_k